# PRDetect — GATv2 (Phan 4) tren HC3

Mirror chinh xac cua `gcn.py` — cung so mau, cung seed, cung config.  
Chi khac: dung **GATv2** thay GCN2.

| Split | So mau (giong gcn.py) |
|-------|----------------------|
| Train | 8,000 |
| Val   | 1,000 |
| Test  | 1,000 |

> **Chon GPU T4** trong Settings + bat **Internet ON** truoc khi chay!

## Buoc 1: Cai dat thu vien

In [ ]:
!pip install torch-geometric transformers datasets -q
!python -m spacy download en_core_web_sm -q
print('Cai dat hoan tat!')

In [ ]:
import torch, torch_geometric
print(f'PyTorch : {torch.__version__}')
print(f'PyG     : {torch_geometric.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Buoc 2: Tai HC3 — Dung dung 8000 Train / 1000 Val / 1000 Test

Tai truc tiep file JSONL tu HuggingFace (tranh loi `trust_remote_code` cua version moi).

In [ ]:
import random, json as jsonlib

# ---- CACH 1: datasets voi data_files URL ----
try:
    from datasets import load_dataset
    print('Dang tai HC3 qua data_files URL...')
    hc3_raw = load_dataset(
        'json',
        data_files='https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl',
        split='train'
    )
    METHOD = 'datasets'
    print(f'Thanh cong! {len(hc3_raw)} QA pairs, cac truong: {list(hc3_raw[0].keys())}')

except Exception as e1:
    # ---- CACH 2: wget + doc tay ----
    print(f'Cach 1 that bai: {e1}')
    print('Thu wget...')
    import subprocess
    subprocess.run([
        'wget', '-q', '-O', '/tmp/hc3_all.jsonl',
        'https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl'
    ], check=True)
    hc3_raw = []
    with open('/tmp/hc3_all.jsonl', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                hc3_raw.append(jsonlib.loads(line))
    METHOD = 'manual'
    print(f'Thanh cong! {len(hc3_raw)} QA pairs')

In [ ]:
def convert_hc3(data):
    """
    Chuyen HC3 sang format PRDetect:
      human_answers   -> label=1 (human)
      chatgpt_answers -> label=0 (AI)
    """
    texts, labels = [], []
    for item in data:
        for ans in item['human_answers']:
            if ans and len(ans.strip()) > 30:
                texts.append(ans.strip())
                labels.append(1)    # human = 1
        for ans in item['chatgpt_answers']:
            if ans and len(ans.strip()) > 30:
                texts.append(ans.strip())
                labels.append(0)    # AI = 0
    return texts, labels


all_texts, all_labels = convert_hc3(hc3_raw)
print(f'Tong mau sau khi convert: {len(all_texts)}')
print(f'Human: {sum(1 for l in all_labels if l==1)}, AI: {sum(1 for l in all_labels if l==0)}')

In [ ]:
# ----------------------------------------------------------------
# Chia CHINH XAC 8000 / 1000 / 1000 — giong hc3_train/val/test.json
# ----------------------------------------------------------------
N_TRAIN = 8000
N_VAL   = 1000
N_TEST  = 1000

combined = list(zip(all_texts, all_labels))
random.seed(2024)           # Giong gcn.py
random.shuffle(combined)

# Dam bao du mau
assert len(combined) >= N_TRAIN + N_VAL + N_TEST, (
    f'Khong du mau! Can {N_TRAIN+N_VAL+N_TEST}, co {len(combined)}'
)

train_texts  = [t for t,_ in combined[:N_TRAIN]]
train_labels = [l for _,l in combined[:N_TRAIN]]
val_texts    = [t for t,_ in combined[N_TRAIN:N_TRAIN+N_VAL]]
val_labels   = [l for _,l in combined[N_TRAIN:N_TRAIN+N_VAL]]
test_texts   = [t for t,_ in combined[N_TRAIN+N_VAL:N_TRAIN+N_VAL+N_TEST]]
test_labels  = [l for _,l in combined[N_TRAIN+N_VAL:N_TRAIN+N_VAL+N_TEST]]

h_tr = sum(1 for l in train_labels if l==1)
m_tr = sum(1 for l in train_labels if l==0)
print(f'Train: {len(train_texts):,} (human={h_tr}, AI={m_tr})  <- giong gcn.py')
print(f'Val  : {len(val_texts):,}                               <- giong gcn.py')
print(f'Test : {len(test_texts):,}                              <- giong gcn.py')

## Buoc 3: Xay dung Do thi Cu phap (SpaCy + RoBERTa)

Giong `building_graph.py` goc.

In [ ]:
import spacy, time
import torch
from transformers import RobertaTokenizer, RobertaModel
from tqdm.notebook import tqdm

print('Nap SpaCy + RoBERTa...')
nlp               = spacy.load('en_core_web_sm')
device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
roberta_model     = RobertaModel.from_pretrained('roberta-base').to(device)
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base', do_lower_case=False)
roberta_model.eval()
print(f'Loaded tren: {device}')

In [ ]:
def build_graph_data(texts, labels, desc='Building'):
    all_token_embeddings = []
    all_edge_index       = []
    valid_labels         = []
    errors = 0

    for idx, text in enumerate(tqdm(texts, desc=desc)):
        try:
            doc    = nlp(text)
            tokens = [token.text for token in doc]
            if len(tokens) < 3:
                errors += 1; continue

            max_length = 512
            chunks     = [tokens[i:i+max_length] for i in range(0, len(tokens), max_length)]
            chunk_out  = []
            for chunk in chunks:
                token_ids = roberta_tokenizer.convert_tokens_to_ids(chunk)
                input_ids = torch.tensor(token_ids).unsqueeze(0).to(device)
                with torch.no_grad():
                    output = roberta_model(input_ids)
                chunk_out.append(output.last_hidden_state[0].cpu())
            embeddings = torch.cat(chunk_out, dim=0)

            edge0 = [word.i      for word in doc]
            edge1 = [word.head.i for word in doc]
            edge_index = torch.tensor([edge0, edge1], dtype=torch.long)

            all_token_embeddings.append(embeddings)
            all_edge_index.append(edge_index)
            valid_labels.append(labels[idx])
        except Exception:
            errors += 1

    y = torch.tensor(valid_labels, dtype=torch.float32)
    print(f'  {desc}: OK={len(valid_labels)}, Errors={errors}')
    return {'all_token_embeddings': all_token_embeddings,
            'all_edge_index':       all_edge_index,
            'y':                    y}


t0 = time.time()
print('[1/3] Building TRAIN (8000 mau)...')
hc3_train_g = build_graph_data(train_texts, train_labels, 'HC3 Train')
print('[2/3] Building VAL (1000 mau)...')
hc3_val_g   = build_graph_data(val_texts,   val_labels,   'HC3 Val  ')
print('[3/3] Building TEST (1000 mau)...')
hc3_test_g  = build_graph_data(test_texts,  test_labels,  'HC3 Test ')
print(f'Hoan tat trong {(time.time()-t0)/60:.1f} phut!')

## Buoc 4: Dinh nghia GATv2 (Phan 4 Cai Tien)

- **[1]** Trong so dong `alpha_ij` — GCN2 dung trong so co dinh theo bac node  
- **[2]** Loc nhieu doi khang — node bi tan cong bi gan `alpha~0`  
- **[3]** Multi-head — 4 heads hoc 4 kieu tuong quan cu phap  
- **[4]** ELU + tach `attn_dropout` vs `feature dropout` + LayerNorm

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data

class GATv2Model(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 heads=4, dropout=0.5, attn_dropout=0.1):
        super(GATv2Model, self).__init__()
        self.conv1 = GATv2Conv(
            in_channels=input_dim, out_channels=hidden_dim,
            heads=heads, dropout=attn_dropout,
            concat=True, add_self_loops=True, share_weights=False
        )
        self.conv2 = GATv2Conv(
            in_channels=hidden_dim * heads, out_channels=output_dim,
            heads=1, dropout=attn_dropout,
            concat=False, add_self_loops=True, share_weights=False
        )
        self.fc      = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.norm1   = nn.LayerNorm(hidden_dim * heads)
        self.norm2   = nn.LayerNorm(output_dim)

    def forward(self, data, return_attention=False):
        x, edge_index = data.x, data.edge_index
        if return_attention:
            x, (_, alpha1) = self.conv1(x, edge_index, return_attention_weights=True)
        else:
            x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.elu(x)
        x = self.dropout(x)
        if return_attention:
            x, (_, alpha2) = self.conv2(x, edge_index, return_attention_weights=True)
        else:
            x = self.conv2(x, edge_index)
        x = self.norm2(x)
        x = self.dropout(x)
        x = self.fc(x)
        x = torch.mean(x, dim=0, keepdim=True)
        if return_attention:
            return torch.sigmoid(x), (alpha1, alpha2)
        return torch.sigmoid(x)


tmp = GATv2Model(768, 64, 64, heads=4)
print(f'GATv2Model — params: {sum(p.numel() for p in tmp.parameters()):,}')
del tmp

## Buoc 5: Cau hinh — Giong het gcn.py (seed=2024, lr=0.0001, epochs=40)

In [ ]:
import torch.optim as optim
import numpy as np
from datetime import datetime

seed         = 2024
dataset_name = 'hc3'

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Tat ca thong so giong gcn.py
input_dim  = 768
hidden_dim = 64
output_dim = 64
heads      = 4
dropout    = 0.5
attn_drop  = 0.1
lr         = 0.0001
epochs     = 40

model = GATv2Model(
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim,
    heads=heads, dropout=dropout, attn_dropout=attn_drop
).to(device)

optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.BCELoss()
SAVE_PATH = f'/kaggle/working/{dataset_name}_gatv2_model_{seed}.pth'

print(f'Device : {device}')
print(f'Params : {sum(p.numel() for p in model.parameters()):,}')

## Buoc 6: Huan luyen — Giong het gcn.py (patience=3)

In [ ]:
train_len = len(hc3_train_g['y'])
val_len   = len(hc3_val_g['y'])

train_loss_hist = []
val_loss_hist   = []
train_acc_hist  = []
val_acc_hist    = []
val_max_acc     = -1
tag             = 3     # patience = 3, giong gcn.py

print(f'Train: {train_len} | Val: {val_len}')
print('=' * 60)

t_start = time.time()

for epoch in range(epochs):

    # ===== Training =====
    model.train()
    epoch_loss = 0.0
    correct_predictions = 0

    for i in tqdm(range(train_len), desc=f'epoch: {epoch+1}, Training', leave=False):
        data = Data(
            x          = hc3_train_g['all_token_embeddings'][i],
            edge_index = hc3_train_g['all_edge_index'][i],
            y          = hc3_train_g['y'][i]
        ).to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss    = criterion(outputs, data.y.float().view(-1, 1))
        loss.backward()
        optimizer.step()
        epoch_loss          += loss.item()
        predictions          = (outputs >= 0.5).long()
        correct_predictions += (predictions == data.y.view(-1, 1)).sum().item()

    epoch_loss /= train_len
    epoch_acc   = correct_predictions / train_len
    train_loss_hist.append(epoch_loss)
    train_acc_hist.append(epoch_acc)
    print(f'epoch: {epoch+1}, train_loss: {epoch_loss:.4f}, train_acc: {epoch_acc:.4f}')

    # ===== Validation =====
    model.eval()
    epoch_loss = 0.0
    correct_predictions = 0

    with torch.no_grad():
        for i in tqdm(range(val_len), desc=f'epoch: {epoch+1}, Validation', leave=False):
            data = Data(
                x          = hc3_val_g['all_token_embeddings'][i],
                edge_index = hc3_val_g['all_edge_index'][i],
                y          = hc3_val_g['y'][i]
            ).to(device)
            outputs = model(data)
            loss    = criterion(outputs, data.y.float().view(-1, 1))
            epoch_loss          += loss.item()
            predictions          = (outputs >= 0.5).long()
            correct_predictions += (predictions == data.y.view(-1, 1)).sum().item()

    epoch_loss /= val_len
    epoch_acc   = correct_predictions / val_len
    val_loss_hist.append(epoch_loss)
    val_acc_hist.append(epoch_acc)
    print(f'epoch: {epoch+1}, val_loss:   {epoch_loss:.4f}, val_acc:   {epoch_acc:.4f}')

    # ===== Save best + Early stopping (patience=3, giong gcn.py) =====
    if epoch_acc >= val_max_acc:
        val_max_acc = epoch_acc
        tag = 3
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  -> Best model saved (val_acc={val_max_acc:.4f})')
    else:
        tag -= 1
        if tag == 0:
            print(f'Early stopping tai epoch {epoch+1}')
            break

elapsed = time.time() - t_start
print(f'\nElapsed: {elapsed:.1f}s ({elapsed/60:.1f} min)')
print(f'Best val_acc: {val_max_acc:.4f}')

## Buoc 7: Testing — Giong het gcn.py

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score

test_len = len(hc3_test_g['y'])

test_model = GATv2Model(
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim,
    heads=heads, dropout=dropout, attn_dropout=attn_drop
).to(device)
test_model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
test_model.eval()

test_loss = 0.0
correct_predictions = 0
test_pres = []

t0 = time.time()
with torch.no_grad():
    for i in tqdm(range(test_len), desc='Test'):
        data = Data(
            x          = hc3_test_g['all_token_embeddings'][i],
            edge_index = hc3_test_g['all_edge_index'][i],
            y          = hc3_test_g['y'][i]
        ).to(device)
        outputs = test_model(data)
        test_pres.append(outputs.item())
        loss = criterion(outputs, data.y.float().view(-1, 1))
        test_loss           += loss.item()
        predictions          = (outputs >= 0.5).long()
        correct_predictions += (predictions == data.y.view(-1, 1)).sum().item()

print(f'Elapsed time: {time.time()-t0:.1f} seconds')

y_pred    = [1 if p >= 0.5 else 0 for p in test_pres]
y_true    = hc3_test_g['y'].view(-1, 1)
test_loss /= test_len
test_acc   = correct_predictions / test_len
test_f1    = f1_score(y_true, y_pred)
auc        = roc_auc_score(hc3_test_g['y'], test_pres)

# Giong gcn.py
print(f'test_loss: {test_loss:.4f}, test_acc: {test_acc:.4f}, test_f1: {test_f1:.4f}')
print(auc)

In [ ]:
# Ghi ket qua — giong gcn.py
result_line = (
    f'hc3_test\t acc: {test_acc:.4f}\t auc: {auc:.4f}\t'
    f' f1: {test_f1:.4f}\t seed: {seed}\t model: gatv2_{dataset_name}\t'
    f'{datetime.now()}\n'
)
with open('/kaggle/working/test_result.txt', 'a', encoding='utf-8') as w:
    w.write(result_line)
print('Da ghi vao /kaggle/working/test_result.txt')
print(result_line)

## Buoc 8: Bieu do Learning Curves

In [ ]:
import matplotlib.pyplot as plt

ep = range(1, len(train_acc_hist) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep, train_acc_hist, 'b--', alpha=0.6, label='Train Acc')
axes[0].plot(ep, val_acc_hist,   'b-',  lw=2,     label='Val Acc')
axes[0].axhline(y=test_acc, color='r', linestyle=':', lw=1.5, label=f'Test Acc={test_acc:.4f}')
axes[0].set_title('GATv2 — Accuracy (HC3, 8000 train)', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, train_loss_hist, 'r--', alpha=0.6, label='Train Loss')
axes[1].plot(ep, val_loss_hist,   'r-',  lw=2,     label='Val Loss')
axes[1].set_title('GATv2 — BCE Loss (HC3, 8000 train)', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('BCE Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(
    f'GATv2 (Phan 4) | HC3: 8000 train / 1000 val / 1000 test\n'
    f'Acc={test_acc:.4f}  AUC={auc:.4f}  F1={test_f1:.4f}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/kaggle/working/gatv2_hc3_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Bieu do: /kaggle/working/gatv2_hc3_curves.png')